# Lora 实战

## Step1 导入相关包

In [1]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

## Step2 加载数据集

In [2]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [3]:
ds[:3]

{'output': ['以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
  '4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。',
  '朱利叶斯·凯撒，又称尤利乌斯·恺撒（Julius Caesar）是古罗马的政治家、军事家和作家。他于公元前44年3月15日被刺杀。 \n\n根据历史记载，当时罗马元老院里一些参议员联合起来策划了对恺撒的刺杀行动，因为他们担心恺撒的统治将给罗马共和制带来威胁。在公元前44年3月15日（又称“3月的艾达之日”），恺撒去参加元老院会议时，被一群参议员包围并被攻击致死。据记载，他身中23刀，其中一刀最终致命。'],
 'input': ['', '输入：4/16', ''],
 'instruction': ['保持健康的三个提示。', '解释为什么以下分数等同于1/4', '朱利叶斯·凯撒是如何死亡的？']}

In [4]:
# print(len("以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。"))

## Step3 数据集预处理

In [5]:
tokenizer = AutoTokenizer.from_pretrained("../../model/meta-llama/Llama-3.2-1B-Instruct")
tokenizer

PreTrainedTokenizerFast(name_or_path='../../model/meta-llama/Llama-3.2-1B-Instruct', vocab_size=128000, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|eot_id|>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128005: AddedToken("<|reserved_special_t

In [6]:
# tokenizer.padding_side = "right"  # 一定要设置padding_side为right，否则batch大于1时可能不收敛

In [7]:
tokenizer.pad_token_id = 128002

In [8]:
def process_func(example):
    MAX_LENGTH = 256    # Llama分词器会将一个中文字切分为多个token，因此需要放开一些最大长度，保证数据的完整性
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ", add_special_tokens=False)
    response = tokenizer(example["output"], add_special_tokens=False)
    input_ids = instruction["input_ids"] + response["input_ids"] + [tokenizer.eos_token_id]
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.eos_token_id]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [9]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26858
})

In [10]:
# print(tokenized_ds[0]["input_ids"])

In [11]:
# tokenizer("abc " + tokenizer.eos_token)

In [12]:
# tokenizer.decode(tokenized_ds[0]["input_ids"])

In [13]:
# tokenizer("呀", add_special_tokens=False) # Llama分词器会将一个中文字切分为多个token，因此需要放开一些最大长度，保证数据的完整性

In [14]:
# tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

## Step4 创建模型

In [15]:
import torch
# 多卡情况，可以去掉device_map="auto"，否则会将模型拆开
model = AutoModelForCausalLM.from_pretrained("../../model/meta-llama/Llama-3.2-1B-Instruct", low_cpu_mem_usage=True, torch_dtype=torch.bfloat16, device_map="auto")

`torch_dtype` is deprecated! Use `dtype` instead!


In [16]:
model.dtype

torch.bfloat16

## Lora

### PEFT Step1 配置文件

In [17]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(task_type=TaskType.CAUSAL_LM,)
config

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules=None, exclude_modules=None, lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None)

### PEFT Step2 创建模型

In [18]:
model = get_peft_model(model, config)

In [19]:
config

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='../../model/meta-llama/Llama-3.2-1B-Instruct', revision=None, inference_mode=False, r=8, target_modules={'v_proj', 'q_proj'}, exclude_modules=None, lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None)

In [20]:
for name, param in model.named_parameters():
    print(name, param.size(), param.dtype)

base_model.model.model.embed_tokens.weight torch.Size([128256, 2048]) torch.bfloat16
base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight torch.Size([2048, 2048]) torch.bfloat16
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.Size([8, 2048]) torch.float32
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.Size([2048, 8]) torch.float32
base_model.model.model.layers.0.self_attn.k_proj.weight torch.Size([512, 2048]) torch.bfloat16
base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight torch.Size([512, 2048]) torch.bfloat16
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight torch.Size([8, 2048]) torch.float32
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight torch.Size([512, 8]) torch.float32
base_model.model.model.layers.0.self_attn.o_proj.weight torch.Size([2048, 2048]) torch.bfloat16
base_model.model.model.layers.0.mlp.gate_proj.weight torch.Size([8192, 2048]) torch.

In [21]:
# model.enable_input_require_grads() # 开启梯度检查点时，要执行该方法

In [ ]:
# model = model.half()  # 当整个模型都是半精度时，需要将adam_epsilon调大，默认1e-8
# torch.tensor(1e-8).half() 

In [ ]:
model = model.bfloat16() 
torch.tensor(1e-8).bfloat16()

tensor(1.0012e-08, dtype=torch.bfloat16)

In [24]:
for name, param in model.named_parameters():
    print(name, param.size(), param.dtype)

base_model.model.model.embed_tokens.weight torch.Size([128256, 2048]) torch.bfloat16
base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight torch.Size([2048, 2048]) torch.bfloat16
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.Size([8, 2048]) torch.bfloat16
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.Size([2048, 8]) torch.bfloat16
base_model.model.model.layers.0.self_attn.k_proj.weight torch.Size([512, 2048]) torch.bfloat16
base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight torch.Size([512, 2048]) torch.bfloat16
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight torch.Size([8, 2048]) torch.bfloat16
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight torch.Size([512, 8]) torch.bfloat16
base_model.model.model.layers.0.self_attn.o_proj.weight torch.Size([2048, 2048]) torch.bfloat16
base_model.model.model.layers.0.mlp.gate_proj.weight torch.Size([8192, 2048]) to

In [25]:
model.print_trainable_parameters()

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


In [26]:
# from torch.utils.data import DataLoader

In [27]:
# dl = DataLoader(tokenized_ds, batch_size=2, collate_fn=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True))

In [28]:
# ipt = next(enumerate(dl))[1]

In [29]:
# ipt

In [30]:
# model(**ipt.to("cuda"))

## Step5 配置训练参数

In [ ]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    logging_steps=10,
    num_train_epochs=1,
    adam_epsilon=1e-8, # fp16需要增大epsilon
)

## Step6 创建训练器

In [32]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds.select(range(6000)),
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

/tmp/ipykernel_22380/2462369116.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


## Step7 模型训练

In [33]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128002}.


Step,Training Loss
10,2.226100
20,2.214900
30,2.153900
40,2.260600
50,2.063200
60,2.101800


KeyboardInterrupt: 

## Step8 模型推理

In [ ]:
model.eval()
ipt = tokenizer("Human: {}\n{}".format("你好", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=512, do_sample=True, eos_token_id=tokenizer.eos_token_id)[0], skip_special_tokens=True)